In [2]:
#| default_exp lawa

In [3]:
#| hide
import nbdev; nbdev.nbdev_export()

In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

In [5]:
#| export
from os import getenv
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from front.common import process_seq

model_path = getenv("MODEL")

In [6]:
model_path = 'large/poetry'
model_path = 'large/pelevin'

# loss 1.43 for llama 3.2 1B
model_path = 'lawa'

In [7]:
#| export
full_path = f'./models/{model_path}'
tokenizer = AutoTokenizer.from_pretrained(full_path)
model = LLM(model=full_path, dtype="bfloat16", device="cuda", gpu_memory_utilization=0.33)

INFO 09-29 07:08:47 config.py:1652] Downcasting torch.float32 to torch.bfloat16.
INFO 09-29 07:08:47 llm_engine.py:226] Initializing an LLM engine (v0.6.1.dev238+ge2c6e0a82) with config: model='./models/large/pelevin', speculative_config=None, tokenizer='./models/large/pelevin', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=./models/large/pelevin, use_v

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


DEBUG 09-29 07:08:48 parallel_state.py:937] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://192.168.1.4:55587 backend=nccl
INFO 09-29 07:08:48 model_runner.py:1014] Starting to load model ./models/large/pelevin...


[W929 07:08:48.820735199 socket.cpp:697] [c10d] The client socket cannot be initialized to connect to [bbb]:55587 (errno: 97 - Address family not supported by protocol).


Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


/usr/local/lib/python3.12/dist-packages/vllm/model_executor/model_loader/weight_utils.py:424: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(bin_file, map_

INFO 09-29 07:08:50 model_runner.py:1025] Loading model weights took 1.4419 GB
INFO 09-29 07:08:51 gpu_executor.py:122] # GPU blocks: 2034, # CPU blocks: 1456
INFO 09-29 07:08:52 model_runner.py:1329] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 09-29 07:08:52 model_runner.py:1333] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 09-29 07:09:07 model_runner.py:1456] Graph capturing finished in 15 secs.


In [8]:
#sum(p.numel() for p in model.model.parameters())

In [9]:
#| export
stop_tokens = ['<|endoftext|>','|eot_id|','<|end_of_text|>','<|eot_id|>']
stop_token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in stop_tokens]
stop_token_ids = [id for sublist in stop_token_ids for id in sublist if len(sublist) == 1]

def create_token_blocker(tokenizer, blocked_tokens):
    blocked_token_ids = set()
    for token in blocked_tokens:
        blocked_token_ids.update(tokenizer.encode(token, add_special_tokens=False))
    
    def token_blocker(input_ids, scores):
        if scores.dim() == 2:
            scores[:, list(blocked_token_ids)] = -float('inf')
        elif scores.dim() == 1:
            scores[list(blocked_token_ids)] = -float('inf')
        else:
            raise ValueError(f"Unexpected score tensor shape: {scores.shape}")
        return scores
    
    return token_blocker
    
def get_sampling_params(tokenizer, length: int, num_samples: int, allow_linebreak: bool, temperature: float):
    blocked_tokens = ['[', '(', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(['\n', '\n\n'])
    
    token_blocker = create_token_blocker(tokenizer, blocked_tokens)

    return SamplingParams(
        temperature=temperature,
        max_tokens=length,
        n=num_samples,
        top_p=0.9,
        top_k=-1,
        stop_token_ids=stop_token_ids,
        ignore_eos=True,
        logits_processors=[token_blocker],
        repetition_penalty=2.,
    )

In [10]:
stop_token_ids

[50257]

In [11]:
#| export
def get_sample(prompt: str, length: int, num_samples: int, allow_linebreak: bool, temperature: float = 1.0):
    sampling_params = get_sampling_params(tokenizer, length, num_samples, allow_linebreak, temperature)
    outputs = model.generate(prompt, sampling_params)
    
    generated_sequences = [oo.text for o in outputs for oo in o.outputs]
    return process_seq(generated_sequences)


In [12]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

DEBUG 09-29 07:09:08 llm_engine.py:1328] Stopping remote worker execution loop.


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  3.16it/s, est. speed input: 28.53 toks/s, output: 633.92 toks/s]

CPU times: user 294 ms, sys: 28.2 ms, total: 322 ms
Wall time: 321 ms


[' – губернатор…» Но Николай Николаевич уже продолжал: «Все равно. Какая все это чушь и избитость! Это надо пережить раз навсегда».',
 ' с помощью пипетки – коварный азиатский диктатор!» Федор и сам не заметил тогда своей откровенности. Теперь он точно знал: в этом виноват профессор Киромцев!',
 ' – козел. Твоя автобиография произвела впечатление… Больше десяти страниц в час бегал! Спрашивал: это нормально?',
 ' – обезьяний Сталин». Тут не хочется говорить о достоинствах той или иной мысли.']